# 01: データセットの構築

UTKFaceの顔画像からArcFaceの顔ベクトルを抽出し，`data/processed/` に学習用・検証用として
保存する．抽出した顔ベクトルは `02_train_vae_anonymizer.ipynb` でAnonymizer(VAE)の学習に使用する．

デモ用として `data/utkface_sample/` に500枚のサンプル画像を同梱している．UTKFace全件
(23,000枚以上)で構築し直す場合は，[UTKFaceの配布ページ](https://susanqq.github.io/UTKFace/)
から取得して `data/UTKFace/` に配置し，`IMAGE_DIR` を書き換えること．

なお，`data/processed/` には全件から抽出済みの顔ベクトルを同梱しているため，本notebookを
実行しなくても `02_train_vae_anonymizer.ipynb` 以降をすぐに試すことができる．


In [1]:
import os
import sys

# notebooks/ から見て1階層上がリポジトリルート
REPO_ROOT = os.path.dirname(os.getcwd())
sys.path.insert(0, os.path.join(REPO_ROOT, "src"))
os.chdir(REPO_ROOT)

import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

_jp_font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
if os.path.exists(_jp_font_path):
    fm.fontManager.addfont(_jp_font_path)
    plt.rcParams["font.family"] = "Noto Sans CJK JP"
    plt.rcParams["axes.unicode_minus"] = False


In [2]:
from identity_anonymizer.faceswap import load_ghost_models
from identity_anonymizer.data import build_face_embedding_dataset, list_utkface_image_paths

models = load_ghost_models()


/home/yryo1005/.conda/envs/py39_dex_ghost_2/lib/python3.9/site-packages/kornia/augmentation/augmentation.py:1830: DeprecationWarning: GaussianBlur is no longer maintained and will be removed from the future versions. Please use RandomGaussianBlur instead.
  warnings.warn(


input mean and std: 127.5 127.5
find model: /home/yryo1005/workspace/identity-anonymizer/weights/ghost/antelope/glintr100.onnx recognition
find model: /home/yryo1005/workspace/identity-anonymizer/weights/ghost/antelope/scrfd_10g_bnkps.onnx detection
set det-size: (640, 640)
loading /home/yryo1005/workspace/identity-anonymizer/weights/ghost/2d106det 0
input mean and std: 127.5 127.5
find model: /home/yryo1005/workspace/identity-anonymizer/weights/ghost/antelope/glintr100.onnx recognition
find model: /home/yryo1005/workspace/identity-anonymizer/weights/ghost/antelope/scrfd_10g_bnkps.onnx detection
set det-size: (640, 640)


[10:59:36] ../src/nnvm/legacy_json_util.cc:208: Loading symbol saved by previous version v1.5.0. Attempting to upgrade...
[10:59:36] ../src/nnvm/legacy_json_util.cc:216: Symbol successfully upgraded!
[10:59:36] ../src/base.cc:79: cuDNN lib mismatch: linked-against version 8302 != compiled-against version 8101.  Set MXNET_CUDNN_LIB_CHECKING=0 to quiet this warning.


In [3]:
IMAGE_DIR = "data/utkface_sample"  # UTKFace全件を使う場合は "data/UTKFace" 等に変更する
OUTPUT_DIR = "data/processed_demo"  # 同梱済みの data/processed を上書きしないよう別ディレクトリに保存する

image_paths = list_utkface_image_paths(IMAGE_DIR)
print(f"対象画像数: {len(image_paths)}")


対象画像数: 500


In [4]:
train_embeddings, test_embeddings = build_face_embedding_dataset(
    image_dir=IMAGE_DIR,
    models=models,
    output_dir=OUTPUT_DIR,
    test_size=0.1,
    seed=0,
)

print(f"学習用: {train_embeddings.shape}, 検証用: {test_embeddings.shape}")


学習用: (420, 512), 検証用: (47, 512)
